> **How to use this notebook.** Step 4 of 6. Needs: the runs of 03 (`RESULTS_DIR/<LABEL>/`) and the DEM (maps). On Colab: the same Drive folders as 02 and 03.
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit, 06_sensitivity (Jupyter or Google Colab). Code: the model (`code/`, CODE_VERSION 2026-11-09-v17); configuration: `config/v16_final/config_full_audited.yaml` (the configuration of the final runs; the same values as `config/config_full_audited.yaml`). Full guide: [notebooks/HOW_TO_USE.md](HOW_TO_USE.md).

# 04 Analysis: tables, maps, centrality, network need

Reads the runs of `03_scenarios.ipynb` (`RESULTS_DIR/<LABEL>/`, the model's code) and uses the analysis tools in
`code/tools/`. Defaults fit the `quick` preset (`NB_PRESET=demo` for the demo runs).

1. island and per-tower-class tables;
2. maps of man-hours per nuraghe: baseline and the headline MOST_REALISTIC_AUDITED_LIT_P10 (and their ratio);
3. centrality of the help networks (`tools/centrality.py`: cooperation = man-hours given, communication = requests),
   top-20 communities and maps;
4. network need (`tools/network_need.py`): each nuraghe's build time with the network vs the same seeds and work
   parties alone (MOST_REALISTIC_P10 vs MOST_REALISTIC_alone_P10);
5. (the PDF report builder of the project is not included in this repository).

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK`,
`NB_INSTALL` (0 = never install) override the defaults (for headless runs with `jupyter nbconvert --execute`).
`CONFIG` ("auto") is `config/v16_final/config_full_audited.yaml`, the configuration of the final runs: the same
values as the model's configuration `config/config_full_audited.yaml` (audited parameters, minimum haul
`transport.min_haul_flatm: 15`, near-site rule `quarry.near_site_fixed_face_m: 1000`); only `cache_dir` / `output_dir` differ.

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/v16_final/config_full_audited.yaml,
                                                         # the configuration of the final runs)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = os.environ.get("NB_INSTALL", "1") == "1"      # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

LABEL = os.environ.get("NB_PRESET", "quick")             # results subfolder written by 03 (= its preset / LABEL)
MAP_SCENARIOS = ["baseline", "scenarios_alliance/MOST_REALISTIC_AUDITED_LIT_P10"]   # folders under RESULTS_DIR/LABEL
NETWORK_SCENARIOS = ["A5_full", "MOST_REALISTIC_AUDITED_LIT_P10"]   # alliance runs for centrality (last = shown)
NEED_PAIRS = ["MOST_REALISTIC_P10:MOST_REALISTIC_alone_P10"]


## 1. Setup

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."

## 2. Tables
Island totals (sum over the sites in the run), then the median man-hours and build years per tower class.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
BASE = os.path.join(RESULTS_DIR, LABEL)                  # the runs of 03_scenarios
assert os.path.isdir(BASE), f"no runs in {BASE}: run 03_scenarios.ipynb first (same OUTPUTS_DIR / LABEL)"
tab = repro.island_table(BASE)
display(tab) if "display" in globals() else print(tab.to_string())
rows = []
for f in sorted(glob.glob(os.path.join(BASE, "**", "nuraghe_manhours_runs.csv.gz"), recursive=True)):
    d = pd.read_csv(f); s = d.groupby(["UID", "TWR_N"])[["mh_total", "days"]].mean().reset_index()
    g = s.groupby("TWR_N").agg(sites=("UID", "size"), MH_median=("mh_total", "median"), days_median=("days", "median"))
    g["scenario"] = os.path.relpath(os.path.dirname(f), BASE); rows.append(g.reset_index())
by_twr = pd.concat(rows).pivot(index="TWR_N", columns="scenario", values="MH_median").round(0)
by_twr

## 3. Maps
One dot per nuraghe, coloured by mean man-hours over seeds (log scale), on the island outline from the DEM.

In [ ]:
import rasterio
from matplotlib.colors import LogNorm
dem = os.path.join(DATA_DIR, "dem", "sardinia_dem_GLO30_25832.tif")
outline = None
if os.path.exists(dem):
    with rasterio.open(dem) as r:
        k = 20; z = r.read(1, out_shape=(r.height // k, r.width // k)); T = r.transform
        outline = (z > 0, (T.c, T.c + r.width * T.a, T.f + r.height * T.e, T.f))
fig, axes = plt.subplots(1, len(MAP_SCENARIOS) + 1, figsize=(6 * (len(MAP_SCENARIOS) + 1), 7))
sums = {}
for ax, sc in zip(axes, MAP_SCENARIOS):
    s = pd.read_csv(os.path.join(BASE, sc, "nuraghe_manhours_summary.csv")); sums[sc] = s
    if outline is not None:
        ax.imshow(outline[0], extent=outline[1], cmap="Greys", alpha=0.25)
    m = ax.scatter(s.x, s.y, c=s.mh_mean, s=8, cmap="viridis", norm=LogNorm())
    plt.colorbar(m, ax=ax, shrink=0.6, label="man-hours (mean over seeds)"); ax.set_title(sc); ax.set_aspect("equal"); ax.axis("off")
a, b = MAP_SCENARIOS[:2]
j = sums[a][["UID", "x", "y", "mh_mean"]].merge(sums[b][["UID", "mh_mean"]], on="UID", suffixes=("_a", "_b"))
ax = axes[-1]
if outline is not None:
    ax.imshow(outline[0], extent=outline[1], cmap="Greys", alpha=0.25)
r = np.log2(j.mh_mean_b / j.mh_mean_a)
m = ax.scatter(j.x, j.y, c=r, s=8, cmap="RdBu_r", vmin=-max(abs(r)), vmax=max(abs(r)))
plt.colorbar(m, ax=ax, shrink=0.6, label=f"log2({b.split('/')[-1]} / {a})"); ax.set_title("ratio"); ax.set_aspect("equal"); ax.axis("off")
os.makedirs(os.path.join(BASE, "analysis"), exist_ok=True)
plt.tight_layout(); plt.savefig(os.path.join(BASE, "analysis", "maps_mh.png"), dpi=120); plt.show()

## 4. Centrality of the help networks
`tools/centrality.py` reads the alliance pair files (`runs/alliance_<scenario>/<signature>/seed_NNNN.pairs.csv.gz`)
and writes, per scenario and layer (cooperation, communication), `nodes.csv`, `top20.csv`, `centrality.shp` and
maps. On a 300-site subset the networks are small; the measures are meaningful only on the full island.

In [ ]:
REF = os.path.join(WORK_DIR, "centrality_ref"); os.makedirs(REF, exist_ok=True)
if not os.path.exists(os.path.join(REF, "data")):
    os.symlink(os.path.join(DATA_DIR, "sites"), os.path.join(REF, "data"))
cen = os.path.join(BASE, "centrality")
# centrality.py reads the site names from REF/data/nurnet_nuraghi.dbf; REF = a folder whose data/ links to DATA_DIR/sites.
# --cache BASE: the part files of 03 (BASE/runs) and the haul table; copy the haul table there once.
import shutil; shutil.copy2(os.path.join(WORK_DIR, "cache", "sites_haul.csv"), os.path.join(BASE, "sites_haul.csv"))
wrap = ("import sys; sys.path.insert(0, 'tools'); import centrality as c; c.REF = sys.argv[1]; "
        "sys.argv = ['tools/centrality.py'] + sys.argv[2:]; c.main()")
repro.py(WORK_DIR, ["-c", wrap, REF, *NETWORK_SCENARIOS,
                    "--cache", BASE, "--out-root", BASE, "--out", cen, "--k", "200"], lock=LOCK)
t = pd.read_csv(os.path.join(cen, NETWORK_SCENARIOS[-1], "cooperation", "top20.csv"))
print(NETWORK_SCENARIOS[-1], "cooperation layer, top 10 by man-hours given (OUT_W):"); t[t.measure == "OUT_W"].head(10)

In [ ]:
from IPython.display import Image, display as _d
for p in sorted(glob.glob(os.path.join(cen, NETWORK_SCENARIOS[-1], "cooperation", "map_*.png")))[:2]:
    print(os.path.basename(p)); _d(Image(p, width=520))

## 5. Network need
Per tower class: share of labour received from other communities, build span with the network vs alone, % of
sites whose alone span exceeds 25 years, % where the network halves the time.

In [ ]:
need = os.path.join(BASE, "network_need")
repro.py(WORK_DIR, ["tools/network_need.py", "--root", os.path.join(BASE, "scenarios_alliance"), "--pairs", *NEED_PAIRS, "--out", need], lock=LOCK)
print(open(os.path.join(need, "network_need.txt")).read()[:3000])
pd.read_csv(os.path.join(need, "network_need_by_towers.csv"))

## 6. Report rebuild: not included
The PDF report builder of the project is not part of this repository (it depends on working notes and on output
files that are not public). The tables of the final runs are a separate data input, the results package (see the top-level README); this notebook
computes its tables from your own runs only.

Next: `05_audit.ipynb` (the step-by-step audit), then `06_sensitivity.ipynb`.